# Shortest Paths and String Matching



```{contents}
:local:
:depth: 2
```


Shortest-path algorithms depend on edge assumptions. String-matching algorithms depend on how much information they reuse after a mismatch. In both areas, the right algorithm follows from the structure of the input.


```{index} shortest path; assumptions
```

## Path Assumptions

Use BFS for unweighted shortest paths. Use Dijkstra's algorithm for nonnegative weighted edges. Be careful with negative weights; they require different algorithms.


In [ ]:
using System;

bool weighted = true;
bool hasNegativeWeights = false;

if (!weighted)
{
    Console.WriteLine("Use BFS for shortest paths by edge count.");
}
else if (!hasNegativeWeights)
{
    Console.WriteLine("Dijkstra's algorithm is a reasonable choice.");
}
else
{
    Console.WriteLine("Use an algorithm designed for negative weights.");
}


This decision is not cosmetic. Using the wrong shortest-path algorithm can produce wrong answers.


```{index} string matching
```

## Direct String Matching

A direct scan tries every possible starting position and checks the pattern character by character.


In [ ]:
using System;

string text = "abracadabra";
string pattern = "cad";
Console.WriteLine(NaiveMatch(text, pattern));

int NaiveMatch(string text, string pattern)
{
    for (int start = 0; start <= text.Length - pattern.Length; start++)
    {
        int offset = 0;
        while (offset < pattern.Length && text[start + offset] == pattern[offset])
        {
            offset++;
        }

        if (offset == pattern.Length)
        {
            return start;
        }
    }

    return -1;
}


Direct matching is simple and often good enough for small inputs or one-off searches.


```{index} prefix table
```

## Prefix Information

Prefix-aware algorithms reuse information about the pattern after a mismatch. The first step is to build a table of proper prefix lengths that are also suffixes.


In [ ]:
using System;

string pattern = "ababaca";
int[] prefix = BuildPrefixTable(pattern);
Console.WriteLine(string.Join(", ", prefix));

int[] BuildPrefixTable(string pattern)
{
    int[] prefix = new int[pattern.Length];
    int length = 0;

    for (int i = 1; i < pattern.Length; i++)
    {
        while (length > 0 && pattern[i] != pattern[length])
        {
            length = prefix[length - 1];
        }

        if (pattern[i] == pattern[length])
        {
            length++;
            prefix[i] = length;
        }
    }

    return prefix;
}


The table says how far the pattern can shift without forgetting useful matched-prefix information.


```{index} KMP algorithm
```

## Prefix-Based Matching

A prefix table supports Knuth-Morris-Pratt style matching. The matcher moves through the text without backing up the text index.


In [ ]:
using System;

string text = "ababcabcabababd";
string pattern = "ababd";
Console.WriteLine(KmpMatch(text, pattern));

int KmpMatch(string text, string pattern)
{
    int[] prefix = BuildPrefixTable(pattern);
    int matched = 0;

    for (int i = 0; i < text.Length; i++)
    {
        while (matched > 0 && text[i] != pattern[matched])
        {
            matched = prefix[matched - 1];
        }

        if (text[i] == pattern[matched])
        {
            matched++;
            if (matched == pattern.Length)
            {
                return i - pattern.Length + 1;
            }
        }
    }

    return -1;
}

int[] BuildPrefixTable(string pattern)
{
    int[] prefix = new int[pattern.Length];
    int length = 0;
    for (int i = 1; i < pattern.Length; i++)
    {
        while (length > 0 && pattern[i] != pattern[length]) length = prefix[length - 1];
        if (pattern[i] == pattern[length]) prefix[i] = ++length;
    }
    return prefix;
}


This implementation returns the first match. Production string searching may need culture, case-sensitivity, Unicode, and multiple-match behavior defined carefully.


```{index} algorithm selection
```

## Choosing a String Strategy

Use the simplest correct method unless input size, repeated searches, or latency requirements justify something more complex.


In [ ]:
using System;

int textLength = 200;
int repeatedSearches = 1;

if (textLength < 1000 && repeatedSearches == 1)
{
    Console.WriteLine("A direct scan is probably clear enough.");
}
else
{
    Console.WriteLine("Consider a prefix-based or library search strategy.");
}


```{rubric} Footnotes
```
[^1]: .NET's built-in string methods are usually the first practical choice. Hand-written matchers are for learning, specialized needs, or controlled experiments.
[^2]: KMP is one example of a prefix-aware string algorithm. Boyer-Moore and Rabin-Karp use different ideas.
